# AutoGluon Tabular Training with Custom PyTorch DLC Image

This notebook demonstrates training with a custom Docker image that installs AutoGluon 1.5.0 on the PyTorch DLC base.

Uses the custom image built in `0-build-push/build_and_push.ipynb`.

In [ ]:
import boto3
from datetime import datetime
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import (
    InputData,
    Compute,
    SourceCode,
    OutputDataConfig,
    StoppingCondition,
)
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Configuration
REGION = boto3.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = Session().default_bucket()
S3_PREFIX = "autogluon-tabular"
INSTANCE_TYPE = "ml.m5.2xlarge"
INSTANCE_COUNT = 1
JOB_NAME = f"ag-tabular-pytorch-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"

# Custom image URI
REPO_NAME = "autogluon-custom"
IMAGE_TAG = "ag150-pytorch-dlc"
image_uri = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAME}:{IMAGE_TAG}-training"

print(f"Region: {REGION}")
print(f"Account: {ACCOUNT_ID}")
print(f"Bucket: {BUCKET}")
print(f"Job Name: {JOB_NAME}")
print(f"Image URI: {image_uri}")

## Discover IAM Role

In [ ]:
role = get_execution_role()
print(f"Execution Role: {role}")

## Build Input Data Channels

Upload local config and serving script to S3, and reference the 1-tabular-classification preprocessed data.

In [ ]:
import sagemaker

s3_client = boto3.client("s3")
sess = sagemaker.Session()

# Upload config
config_s3_path = sess.upload_data(
    path="config.yaml",
    bucket=BUCKET,
    key_prefix=f"{S3_PREFIX}/custom-image-pytorch/config",
)
print(f"Config uploaded to: {config_s3_path}")

# Upload serving script
serving_s3_path = sess.upload_data(
    path="../2-inference/serve.py",
    bucket=BUCKET,
    key_prefix=f"{S3_PREFIX}/custom-image-pytorch/serving",
)
print(f"Serving script uploaded to: {serving_s3_path}")

# Reference preprocessed data from 1-tabular-classification experiment
S3_TRAIN = f"s3://{BUCKET}/{S3_PREFIX}/processed/train/"
S3_TEST = f"s3://{BUCKET}/{S3_PREFIX}/processed/test/"

print(f"Training data: {S3_TRAIN}")
print(f"Test data: {S3_TEST}")

## Create ModelTrainer and Launch

Uses SageMaker SDK v3 with custom Docker image built on PyTorch DLC.

In [ ]:
# Define input channels
input_data = [
    InputData(channel_name="train", data_source=S3_TRAIN),
    InputData(channel_name="test", data_source=S3_TEST),
    InputData(channel_name="config", data_source=config_s3_path),
    InputData(channel_name="serving", data_source=serving_s3_path),
]

# Create ModelTrainer
trainer = ModelTrainer(
    training_job_name=JOB_NAME,
    role_arn=role,
    algorithm_specification_training_image=image_uri,
    algorithm_specification_training_input_mode="File",
    input_data_config=input_data,
    resource_config=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=INSTANCE_COUNT,
        volume_size_in_gb=30,
    ),
    source_code=SourceCode(
        source_code_s3_uri=sess.upload_data(
            path="train.py",
            bucket=BUCKET,
            key_prefix=f"{S3_PREFIX}/custom-image-pytorch/source",
        )
    ),
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{BUCKET}/{S3_PREFIX}/custom-image-pytorch/output"
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
)

# Launch training
print(f"\nStarting training job: {JOB_NAME}")
trainer.create()

print(f"\nTraining job ARN: {trainer.arn}")
print(f"Model artifact will be at: {trainer.model_artifacts_s3_uri}")

## Training Job Summary

In [ ]:
# Wait for training to complete
trainer.wait(logs=True)

print(f"\nTraining Status: {trainer.training_job_status}")
print(f"Model Artifact: {trainer.model_artifacts_s3_uri}")